In [1]:
!pip install ase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 8.0 MB/s  0:00:00


# F5BzPA and FBzPA + Au 111


In [ ]:
import itertools, os
import numpy as np
from ase.io import read, write
from ase.geometry import find_mic
from ase.constraints import FixAtoms

# ---------------- user settings ----------------
SLAB_FILE = "/content/CONTCAR_Au"
MOLECULES = {                                   # label: path to optimized molecule
    "F5BzPA": "/content/CONTCAR_F5BzPA",
    "FBzPA":  "/content/CONTCAR_FBzPA",
}
OUT_ROOT    = "/content/POSCAR_PA_on_Au111"
D_AU_O      = 2.20    # target Au-O distance for bound O (Angstrom)
D_PO        = 1.52    # P-O bond length used to build target for P
D_PC        = 1.82    # P-C bond length used to build target for C
H_INTERACT  = 2.90    # height above top Au plane for the non-bonded O in mode (b)
VACUUM_TOP  = 12.0    # vacuum above the highest atom (Angstrom)
N_FREEZE    = 2       # number of bottom Au layers to freeze (0 = none)
H_POLICIES  = ("remove", "keep")  # remove H from bound O (P-O-Au) or keep it
MIN_CONTACT = 1.9     # minimum distance of non-bound atoms to Au after lifting (Angstrom)
MAX_LIFT    = 1.0     # largest upward shift allowed (Angstrom)
LIFT_STEP   = 0.05
# -----------------------------------------------

# ================= slab analysis (done once) =================
slab = read(SLAB_FILE)
pos = slab.positions
ztop = pos[:, 2].max()
top = np.where(pos[:, 2] > ztop - 0.5)[0]
second = np.where((pos[:, 2] < ztop - 0.5) & (pos[:, 2] > ztop - 3.0))[0]
dall = slab.get_all_distances(mic=True)
dt = dall[np.ix_(top, top)]
nn = dt[dt > 0.1].min()
print(f"Top layer atoms: {len(top)}, nearest-neighbor distance: {nn:.3f} A")

xy_center = slab.cell[:2, :2].sum(axis=0) / 2
anchor = top[np.argmin(np.linalg.norm(pos[top, :2] - xy_center, axis=1))]
vec = {j: slab.get_distance(anchor, j, mic=True, vector=True)
       for j in top if j != anchor and slab.get_distance(anchor, j, mic=True) < 1.1 * nn}
nbrs = list(vec.keys())

def site_z(r_inplane):
    return ztop + np.sqrt(max(D_AU_O**2 - r_inplane**2, 0.5**2))

Z_ATOP, Z_BRIDGE, Z_HOLLOW = site_z(0.0), site_z(nn / 2), site_z(nn / np.sqrt(3))

tri = {}   # fcc / hcp triangles containing the anchor atom
for j, k in itertools.combinations(nbrs, 2):
    if slab.get_distance(j, k, mic=True) < 1.1 * nn:
        cen = pos[anchor] + (vec[j] + vec[k]) / 3
        d = pos[second] - cen
        d[:, 2] = 0.0
        dmin = find_mic(d, slab.cell, slab.pbc)[1].min()
        kind = "hcp" if dmin < 0.3 else "fcc"
        tri.setdefault(kind, (j, k, cen))

def atop(i):
    p = pos[i].copy(); p[2] = Z_ATOP; return p

A = atop(anchor)
B = pos[anchor] + vec[nbrs[0]]; B[2] = Z_ATOP
bridge = pos[anchor] + vec[nbrs[0]] / 2; bridge[2] = Z_BRIDGE

site_lookup = {"atop": A, "bridge": bridge}
for kind in tri:
    c = tri[kind][2].copy(); c[2] = Z_HOLLOW
    site_lookup[kind] = c

# layers to freeze
zs = np.sort(np.unique(np.round(pos[:, 2], 1)))
layer_z = []
for z in zs:
    if not layer_z or z - layer_z[-1] > 0.5:
        layer_z.append(z)
fix_idx = [i for i in range(len(slab)) if N_FREEZE > 0 and pos[i, 2] < layer_z[N_FREEZE - 1] + 0.5]

# ================= generic helpers =================
def kabsch(P, Q, w):
    w = np.asarray(w, float)
    cp = (P * w[:, None]).sum(0) / w.sum()
    cq = (Q * w[:, None]).sum(0) / w.sum()
    H = (P - cp).T @ ((Q - cq) * w[:, None])
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, cp, cq

def P_C_targets(O_targets):
    """P above the centroid of the O targets, C above P along +z."""
    T = np.array(O_targets)
    cxy = T[:, :2].mean(0)
    r = np.linalg.norm(T[:, :2] - cxy, axis=1).mean()
    dz = np.sqrt(max(D_PO**2 - r**2, 0.5**2))
    Pt = np.array([cxy[0], cxy[1], T[:, 2].mean() + dz])
    Ct = Pt + np.array([0, 0, D_PC])
    return Pt, Ct

# ================= per-molecule builder =================
def process(label, mol_file):
    out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)
    mol = read(mol_file)
    mol.set_pbc(False)

    # ---- molecule bookkeeping ----
    sym = np.array(mol.get_chemical_symbols())
    iP = int(np.where(sym == "P")[0][0])
    dP = np.linalg.norm(mol.positions - mol.positions[iP], axis=1)
    O_idx = [int(i) for i in np.where(sym == "O")[0] if dP[i] < 2.0]
    iC = int([i for i in np.where(sym == "C")[0] if dP[i] < 2.2][0])
    assert len(O_idx) == 3, f"{label}: expected 3 O on P, found {len(O_idx)} (is the molecule unwrapped?)"

    H_of = {}
    for o in O_idx:
        do = np.linalg.norm(mol.positions - mol.positions[o], axis=1)
        H_of[o] = [int(h) for h in np.where(sym == "H")[0] if do[h] < 1.2]
    oh_O = [o for o in O_idx if H_of[o]]
    po_O = [o for o in O_idx if not H_of[o]]
    assert len(oh_O) == 2 and len(po_O) == 1, f"{label}: expected 2 P-OH and 1 P=O"
    print(f"\n[{label}] P: {iP}  C: {iC}  P-OH O: {oh_O}  P=O O: {po_O}")

    # ---- fitting helpers that depend on this molecule ----
    def fit(idx, targets, weights):
        P = mol.positions[idx]
        Q = np.array(targets)
        R, cp, cq = kabsch(P, Q, weights)
        new = (mol.positions - cp) @ R.T + cq
        rmsd = np.sqrt(np.average(((new[idx] - Q) ** 2).sum(1), weights=weights))
        return new, rmsd

    def clash_metric(new_pos, exclude):
        """Smallest distance between molecule atoms (except those in `exclude`) and slab atoms."""
        tmp = slab.copy()
        m = mol.copy(); m.positions = new_pos
        tmp += m
        D = tmp.get_all_distances(mic=True)
        ns = len(slab)
        rows = [ns + i for i in range(len(mol)) if i not in exclude]
        return D[np.ix_(rows, range(ns))].min()

    def best_assignment(O_list, O_targets, weights_O):
        best = None
        Pt, Ct = P_C_targets(O_targets)
        for perm in itertools.permutations(range(len(O_targets))):
            tg = [O_targets[perm[p]] for p in range(len(O_list))]
            new, rmsd = fit(list(O_list) + [iP, iC], tg + [Pt, Ct], list(weights_O) + [1.0, 1.0])
            if best is None or rmsd < best[1]:
                best = (new, rmsd)
        return best

    # ---- mode builders: return (new_positions, bound_O_list, rmsd) ----
    def mono(site_pos):
        O_b = oh_O[0]
        best = None
        for phi in np.deg2rad(np.arange(0, 360, 30)):
            d_PO = np.array([np.sin(np.deg2rad(109.5)) * np.cos(phi),
                             np.sin(np.deg2rad(109.5)) * np.sin(phi),
                             np.cos(np.deg2rad(109.5))])       # P -> O direction
            Pt = site_pos - D_PO * d_PO
            Ct = Pt + np.array([0, 0, D_PC])
            new, rmsd = fit([O_b, iP, iC], [site_pos, Pt, Ct], [1, 1, 1])
            c = clash_metric(new, {O_b})
            if best is None or c > best[0]:
                best = (c, new, rmsd)
        return best[1], [O_b], best[2]

    def bidentate():
        new, rmsd = best_assignment(oh_O, [A, B], [1, 1])
        return new, list(oh_O), rmsd

    def bi_electro():
        O_b, O_i = oh_O[0], po_O[0]
        tgt_b = A
        tgt_i = B.copy(); tgt_i[2] = ztop + H_INTERACT
        Pt, Ct = P_C_targets([tgt_b, tgt_i])
        new, rmsd = fit([O_b, O_i, iP, iC], [tgt_b, tgt_i, Pt, Ct], [2, 1, 1, 1])
        return new, [O_b], rmsd

    def tridentate(kind):
        j, k, _ = tri[kind]
        T = [atop(anchor), atop(j), atop(k)]
        new, rmsd = best_assignment(O_idx, T, [1, 1, 1])
        return new, list(O_idx), rmsd

    jobs = [(f"mono_{s}", (lambda s=s: mono(site_lookup[s]))) for s in site_lookup]
    jobs.append(("bi_atop-atop", bidentate))
    jobs.append(("bi-electro_atop", bi_electro))
    jobs += [(f"tri_{kd}", (lambda kd=kd: tridentate(kd))) for kd in tri]

    # ---- build and write ----
    summary = []
    for mode, builder in jobs:
        new_pos, bound, rmsd = builder()
        for policy in H_POLICIES:
            drop = {h for o in bound for h in H_of[o]} if policy == "remove" else set()
            exclude = set(bound) | drop

            # lift the molecule until non-bound atoms are at least MIN_CONTACT from Au
            lift = 0.0
            while (clash_metric(new_pos + np.array([0, 0, lift]), exclude) < MIN_CONTACT
                   and lift < MAX_LIFT):
                lift += LIFT_STEP
            placed = new_pos + np.array([0, 0, lift])
            contact = clash_metric(placed, exclude)

            m = mol.copy()
            m.positions = placed
            if drop:
                m = m[[i for i in range(len(m)) if i not in drop]]
            s2 = slab.copy()
            s2 += m
            zmax = s2.positions[:, 2].max()
            c_new = max(slab.cell[2, 2], zmax + VACUUM_TOP)
            cell = s2.cell.array.copy(); cell[2] = [0, 0, c_new]
            s2.set_cell(cell, scale_atoms=False)
            s2.set_constraint(FixAtoms(indices=fix_idx))

            Dm = s2.get_all_distances(mic=True)
            Dno = s2.get_all_distances(mic=False)
            ns = len(slab)
            dmin_slab = Dm[ns:, :ns].min()
            sub, subn = Dm[ns:, ns:], Dno[ns:, ns:]
            img = sub[sub < subn - 0.01]
            dimg = img.min() if img.size else np.inf

            name = f"POSCAR_{label}_{mode}_H{policy}"
            write(os.path.join(out_dir, name), s2, format="vasp", direct=True, vasp5=True, sort=True)
            summary.append((name, rmsd, lift, contact, dmin_slab, dimg))
    return summary

# ================= run all molecules =================
for label, path in MOLECULES.items():
    summ = process(label, path)
    print(f"{'file':42s} {'fit rmsd':>9s} {'lift':>6s} {'min d(non-bound)':>17s} {'min d(all)':>11s} {'min d(image)':>13s}")
    for n, r, lf, c, d1, d2 in summ:
        flag = "  <-- check" if (c < MIN_CONTACT - 0.05 or d2 < 2.0) else ""
        print(f"{n:42s} {r:9.2f} {lf:6.2f} {c:17.2f} {d1:11.2f} {d2:13.2f}{flag}")

Top layer atoms: 16, nearest-neighbor distance: 2.884 A

[F5BzPA] P: 16  C: 6  P-OH O: [17, 18]  P=O O: [19]
file                                        fit rmsd   lift  min d(non-bound)  min d(all)  min d(image)
POSCAR_F5BzPA_mono_atop_Hremove                 0.07   0.00              2.31        2.26          4.70
POSCAR_F5BzPA_mono_atop_Hkeep                   0.07   0.45              1.92        1.92          4.70
POSCAR_F5BzPA_mono_bridge_Hremove               0.07   0.25              1.93        1.93          4.78
POSCAR_F5BzPA_mono_bridge_Hkeep                 0.07   0.40              1.92        1.92          4.78
POSCAR_F5BzPA_mono_fcc_Hremove                  0.07   0.20              1.90        1.90          4.78
POSCAR_F5BzPA_mono_fcc_Hkeep                    0.07   0.65              1.91        1.91          4.78
POSCAR_F5BzPA_mono_hcp_Hremove                  0.07   0.20              1.90        1.90          4.28
POSCAR_F5BzPA_mono_hcp_Hkeep                    0.07   0.65

In [ ]:
import shutil, os
from google.colab import files

OUT_ROOT = "/content/POSCAR_PA_on_Au111"    # output folder from the script (contains one subfolder per molecule)

shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)       # creates /content/POSCAR_PA_on_Au111.zip
n_files = sum(len(fs) for _, _, fs in os.walk(OUT_ROOT))
print(f"Zipped {n_files} files")
files.download(OUT_ROOT + ".zip")

Zipped 32 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil, os

OUT_ROOT = "/content/POSCAR_PA_on_Au111"

shutil.rmtree(OUT_ROOT, ignore_errors=True)          # the folder with all POSCARs
if os.path.exists(OUT_ROOT + ".zip"):
    os.remove(OUT_ROOT + ".zip")                     # the zip file
print("Removed:", OUT_ROOT, "and", OUT_ROOT + ".zip")

Removed: /content/POSCAR_PA_on_Au111 and /content/POSCAR_PA_on_Au111.zip


# PFBT and FBzS + Au 111

In [ ]:
import itertools, os
import numpy as np
from ase.io import read, write
from ase.geometry import find_mic, get_distances
from ase.constraints import FixAtoms

# ---------------- user settings ----------------
SLAB_FILE = "/content/CONTCAR_Au"
MOLECULES = {
    "PFBT": "/content/CONTCAR_PFBT",   # adjust to your actual file names
    "FBzS": "/content/CONTCAR_FBzS",
}
OUT_ROOT   = "/content/POSCAR_thiol_on_Au111"
D_AU_S     = 2.40    # target Au-S distance (Angstrom)
TILTS      = (0.0, 30.0)                     # tilt of the S->C bond from the surface normal (degrees)
PHIS       = tuple(np.arange(0, 360, 60))    # azimuth of the tilt (degrees), used when tilt > 0
PSIS       = tuple(np.arange(0, 360, 30))    # spin about the S->C bond (degrees)
VACUUM_TOP = 12.0
N_FREEZE   = 2
H_POLICIES = ("remove", "keep")  # remove = thiolate (S-Au), keep = intact S-H thiol
# -----------------------------------------------

# ================= slab analysis (done once) =================
slab = read(SLAB_FILE)
pos = slab.positions
ztop = pos[:, 2].max()
top = np.where(pos[:, 2] > ztop - 0.5)[0]
second = np.where((pos[:, 2] < ztop - 0.5) & (pos[:, 2] > ztop - 3.0))[0]
dall = slab.get_all_distances(mic=True)
dt = dall[np.ix_(top, top)]
nn = dt[dt > 0.1].min()
print(f"Top layer atoms: {len(top)}, nearest-neighbor distance: {nn:.3f} A")

xy_center = slab.cell[:2, :2].sum(axis=0) / 2
anchor = top[np.argmin(np.linalg.norm(pos[top, :2] - xy_center, axis=1))]
vec = {j: slab.get_distance(anchor, j, mic=True, vector=True)
       for j in top if j != anchor and slab.get_distance(anchor, j, mic=True) < 1.1 * nn}
nbrs = list(vec.keys())

def z_for_xy(p):
    """Height of S above the top plane so that the nearest top-layer Au is D_AU_S away."""
    d = pos[top] - p
    d[:, 2] = 0.0
    r = find_mic(d, slab.cell, slab.pbc)[1].min()
    return ztop + np.sqrt(max(D_AU_S**2 - r**2, 0.5**2))

def with_z(p):
    q = np.array(p, float); q[2] = z_for_xy(q); return q

tri = {}
for j, k in itertools.combinations(nbrs, 2):
    if slab.get_distance(j, k, mic=True) < 1.1 * nn:
        cen = pos[anchor] + (vec[j] + vec[k]) / 3
        d = pos[second] - cen
        d[:, 2] = 0.0
        dmin = find_mic(d, slab.cell, slab.pbc)[1].min()
        kind = "hcp" if dmin < 0.3 else "fcc"
        tri.setdefault(kind, (j, k, cen))

sites = {"atop": with_z(pos[anchor]),
         "bridge": with_z(pos[anchor] + vec[nbrs[0]] / 2)}
for kind, (j, k, cen) in tri.items():
    sites[kind] = with_z(cen)
    mid = pos[anchor] + vec[j] / 2
    sites[f"bridge-{kind}"] = with_z(mid + 0.5 * (cen - mid))   # bridge shifted toward that hollow

# layers to freeze
zs = np.sort(np.unique(np.round(pos[:, 2], 1)))
layer_z = []
for z in zs:
    if not layer_z or z - layer_z[-1] > 0.5:
        layer_z.append(z)
fix_idx = [i for i in range(len(slab)) if N_FREEZE > 0 and pos[i, 2] < layer_z[N_FREEZE - 1] + 0.5]

# ================= rotation helpers =================
def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def rot_axis(axis, ang):
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    K = skew(axis)
    return np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * (K @ K)

def rot_align(a, b):
    a = a / np.linalg.norm(a); b = b / np.linalg.norm(b)
    v = np.cross(a, b); c = float(np.dot(a, b)); s = np.linalg.norm(v)
    if s < 1e-8:
        if c > 0:
            return np.eye(3)
        ax = np.cross(a, [1, 0, 0])
        if np.linalg.norm(ax) < 1e-6:
            ax = np.cross(a, [0, 1, 0])
        return rot_axis(ax, np.pi)
    K = skew(v)
    return np.eye(3) + K + (K @ K) * ((1 - c) / s**2)

# ================= per-molecule builder =================
def process(label, mol_file):
    out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)
    mol = read(mol_file)
    mol.set_pbc(False)

    sym = np.array(mol.get_chemical_symbols())
    assert (sym == "S").sum() == 1, f"{label}: expected exactly one S"
    iS = int(np.where(sym == "S")[0][0])
    dS = np.linalg.norm(mol.positions - mol.positions[iS], axis=1)
    Cs = [int(i) for i in np.where(sym == "C")[0] if dS[i] < 2.0]
    assert len(Cs) == 1, f"{label}: expected one C bonded to S, found {len(Cs)}"
    iC = Cs[0]
    H_S = [int(h) for h in np.where(sym == "H")[0] if dS[h] < 1.6]
    print(f"\n[{label}] S: {iS}  C: {iC}  H on S: {H_S}")

    policies = H_POLICIES if H_S else ("keep",)
    u0 = mol.positions[iC] - mol.positions[iS]
    summary = []

    for site, spos in sites.items():
        for tilt in TILTS:
            for policy in policies:
                excl = {iS} | (set(H_S) if policy == "remove" else set())
                rows = [i for i in range(len(mol)) if i not in excl]
                best = None
                for phi in (PHIS if tilt > 0 else (0.0,)):
                    t = np.array([np.sin(np.deg2rad(tilt)) * np.cos(np.deg2rad(phi)),
                                  np.sin(np.deg2rad(tilt)) * np.sin(np.deg2rad(phi)),
                                  np.cos(np.deg2rad(tilt))])
                    R1 = rot_align(u0, t)
                    for psi in PSIS:
                        R = rot_axis(t, np.deg2rad(psi)) @ R1
                        newpos = (mol.positions - mol.positions[iS]) @ R.T + spos
                        D = get_distances(newpos[rows], pos, cell=slab.cell, pbc=slab.pbc)[1]
                        c = D.min()
                        if best is None or c > best[0]:
                            best = (c, newpos)
                m = mol.copy()
                m.positions = best[1]
                if policy == "remove":
                    m = m[[i for i in range(len(m)) if i not in set(H_S)]]
                s2 = slab.copy()
                s2 += m
                zmax = s2.positions[:, 2].max()
                c_new = max(slab.cell[2, 2], zmax + VACUUM_TOP)
                cell = s2.cell.array.copy(); cell[2] = [0, 0, c_new]
                s2.set_cell(cell, scale_atoms=False)
                s2.set_constraint(FixAtoms(indices=fix_idx))
                ns = len(slab)
                Dm = s2.get_all_distances(mic=True)
                Dno = s2.get_all_distances(mic=False)
                sub, subn = Dm[ns:, ns:], Dno[ns:, ns:]
                img = sub[sub < subn - 0.01]
                dimg = img.min() if img.size else np.inf
                dAuS = get_distances([spos], pos, cell=slab.cell, pbc=slab.pbc)[1].min()
                name = f"POSCAR_{label}_{site}_tilt{int(tilt)}_H{policy}"
                write(os.path.join(out_dir, name), s2, format="vasp", direct=True, vasp5=True, sort=True)
                summary.append((name, dAuS, best[0], dimg))
    return summary

for label, path in MOLECULES.items():
    summ = process(label, path)
    print(f"{'file':44s} {'d(Au-S)':>8s} {'min d(other-Au)':>16s} {'min d(image)':>13s}")
    for n, d0, d1, d2 in summ:
        flag = "  <-- check" if (d1 < 2.0 or d2 < 2.0) else ""
        print(f"{n:44s} {d0:8.2f} {d1:16.2f} {d2:13.2f}{flag}")

Top layer atoms: 16, nearest-neighbor distance: 2.884 A

[PFBT] S: 11  C: 2  H on S: [12]
file                                          d(Au-S)  min d(other-Au)  min d(image)
POSCAR_PFBT_atop_tilt0_Hremove                   2.40             4.17           inf
POSCAR_PFBT_atop_tilt0_Hkeep                     2.40             2.67           inf
POSCAR_PFBT_atop_tilt30_Hremove                  2.40             4.01           inf
POSCAR_PFBT_atop_tilt30_Hkeep                    2.40             3.15           inf
POSCAR_PFBT_bridge_tilt0_Hremove                 2.40             3.91           inf
POSCAR_PFBT_bridge_tilt0_Hkeep                   2.40             2.33           inf
POSCAR_PFBT_bridge_tilt30_Hremove                2.40             3.67           inf
POSCAR_PFBT_bridge_tilt30_Hkeep                  2.40             2.65           inf
POSCAR_PFBT_fcc_tilt0_Hremove                    2.40             3.71           inf
POSCAR_PFBT_fcc_tilt0_Hkeep                      2.40       

In [ ]:
import shutil, os
from google.colab import files

OUT_ROOT = "/content/POSCAR_thiol_on_Au111"    # output folder from the script (contains one subfolder per molecule)

shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)       # creates /content/POSCAR_thiol_on_Au111.zip
n_files = sum(len(fs) for _, _, fs in os.walk(OUT_ROOT))
print(f"Zipped {n_files} files")
files.download(OUT_ROOT + ".zip")

Zipped 48 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil, os

OUT_ROOT = "/content/POSCAR_thiol_on_Au111"

shutil.rmtree(OUT_ROOT, ignore_errors=True)          # the folder with all POSCARs
if os.path.exists(OUT_ROOT + ".zip"):
    os.remove(OUT_ROOT + ".zip")                     # the zip file
print("Removed:", OUT_ROOT, "and", OUT_ROOT + ".zip")

Removed: /content/POSCAR_PA_on_Au111 and /content/POSCAR_PA_on_Au111.zip


# F5BzPA and FBzPA + ITO

In [2]:
import itertools, os
import numpy as np
from ase.io import read, write
from ase.geometry import find_mic, get_distances
from ase.constraints import FixAtoms

# =====================================================================
# Phosphonic acid (PA) on hydroxylated ITO(222), following the model of
# Paramonov et al., Chem. Mater. 2008, 20, 5131 (Scheme 1):
#   (a) monodentate         : 1 P-O-M bond
#   (b) bidentate + electro.: 1 P-O-M bond + P-OH...O(ITO) hydrogen bond
#   (c) bidentate           : 2 P-O-M bonds
#   (d) tridentate          : 3 P-O-M bonds
# Chemisorption proceeds by condensation with surface hydroxyls,
#   P-OH + HO-M  ->  P-O-M + H2O,
# eliminating up to two water molecules per PA. The two sampled anchor types:
#   "OH": a surface hydroxyl. The P-O takes the position of the hydroxyl O;
#         the hydroxyl (O+H) and the P-OH hydrogen are removed as water.
#   "M" : an under-coordinated metal ion. The P-O binds the open site, no
#         water is eliminated (the acid H stays on that O).
# =====================================================================

# ---------------- user settings ----------------
SLAB_FILE = "/content/CONTCAR"                  # ITO slab
MOLECULES = {                                   # label: path to optimized molecule
    "F5BzPA": "/content/CONTCAR_F5BzPA",
    "FBzPA":  "/content/CONTCAR_FBzPA",
}
OUT_ROOT    = "/content/POSCAR_PA_on_ITO"
METALS      = ("In", "Sn")
D_M_O       = {"In": 2.15, "Sn": 2.05}   # M-O distance for O placed on an open metal site (A)
D_PO        = 1.52    # P-O bond length
D_PC        = 1.82    # P-C bond length
D_HBOND     = 1.70    # target H...O distance for the P-OH...O(ITO) hydrogen bond in mode (b)
OH_CUTOFF   = 1.20    # O-H distance that defines a surface hydroxyl (A)
MO_CUTOFF   = 2.70    # M-O cutoff for coordination / open direction (A)
SURF_DEPTH  = 2.5     # metals within this depth of the highest metal are surface candidates (A)
MIN_OPEN    = 0.35    # minimum |sum of unit vectors to O neighbours| for an exposed cation
MIN_UP      = 0.30    # minimum z component of the open direction
CLEAR_HEAVY = 2.00    # open-site O must be this far from other slab heavy atoms (A)
CLEAR_H     = 1.50    # ... and this far from slab H atoms (A)
N_OH_ANCHORS = 12     # surface hydroxyls kept as anchors (highest first)
N_M_ANCHORS  = 8      # open metal sites kept as anchors (most exposed first)
N_MONO       = 4      # monodentate anchors per anchor type
N_HBOND      = 4      # mode (b) anchors per anchor type
N_PER_CLASS_BI  = 3   # bidentate pairs per water-count class (0, 1, 2 H2O)
N_PER_CLASS_TRI = 2   # tridentate triples per water-count class (0, 1, 2 H2O)
OO_MIN, OO_MAX = 2.3, 4.4   # allowed distance between anchor positions for multidentate modes (A)
VACUUM_TOP  = 12.0
FREEZE_BELOW = 3.0    # freeze slab atoms with z below this value (A); 0 = none
MIN_CONTACT = 1.8     # minimum distance of non-bound molecule atoms to slab after lifting (A)
MAX_LIFT    = 1.0
LIFT_STEP   = 0.05
# -----------------------------------------------

# ================= slab analysis (done once) =================
slab = read(SLAB_FILE)
slab.set_pbc(True)
pos = slab.positions
sym_s = np.array(slab.get_chemical_symbols())
ns = len(slab)
CELL, PBC = slab.cell, slab.pbc
metal_idx = np.where(np.isin(sym_s, METALS))[0]
O_idx_s = np.where(sym_s == "O")[0]
H_idx_s = np.where(sym_s == "H")[0]
heavy_idx = np.where(sym_s != "H")[0]
zmetal = pos[metal_idx, 2].max()
ztop = pos[heavy_idx, 2].max()
print(f"Slab: {slab.get_chemical_formula()}  highest metal z = {zmetal:.2f}  highest heavy atom z = {ztop:.2f}")

Dss = slab.get_all_distances(mic=True)

def mic_vec(a, b):
    return find_mic((b - a)[None, :], CELL, PBC)[0][0]

def unwrap_to(ref, p):
    return ref + mic_vec(ref, p)

# ---- anchor type "OH": surface hydroxyls ----
anchors_OH = []
hyd_O = {}
for h in H_idx_s:
    o = int(O_idx_s[np.argmin(Dss[h, O_idx_s])])
    if Dss[h, o] < OH_CUTOFF:
        hyd_O[o] = int(h)
for o, h in hyd_O.items():
    ms = [int(m) for m in metal_idx if Dss[o, m] < MO_CUTOFF]
    if ms:
        cen = np.mean([unwrap_to(pos[o], pos[m]) for m in ms], axis=0)
        d = pos[o] - cen
        d = d / np.linalg.norm(d) if np.linalg.norm(d) > 1e-6 else np.array([0, 0, 1.0])
    else:
        d = np.array([0, 0, 1.0])
    anchors_OH.append(dict(kind="OH", tag=f"OH{o}", pos=pos[o].copy(), dir=d, leave=[o, h], score=pos[o, 2], nmetal=len(ms)))
anchors_OH.sort(key=lambda a: -a["score"])
anchors_OH = anchors_OH[:N_OH_ANCHORS]

# ---- anchor type "M": exposed, unobstructed under-coordinated cations ----
anchors_M = []
for m in metal_idx:
    if pos[m, 2] < zmetal - SURF_DEPTH:
        continue
    vecs = slab.get_distances(int(m), range(ns), mic=True, vector=True)
    dist = np.linalg.norm(vecs, axis=1)
    nb = [j for j in range(ns) if j != m and sym_s[j] == "O" and dist[j] < MO_CUTOFF]
    if not nb:
        continue
    u = vecs[nb] / dist[nb][:, None]
    ov = -u.sum(axis=0)
    mag = np.linalg.norm(ov)
    if mag < MIN_OPEN:
        continue
    d = ov / mag
    if d[2] < MIN_UP:
        continue
    el = sym_s[m]
    p = pos[m] + D_M_O[el] * d
    dq = np.linalg.norm(find_mic(pos - p, CELL, PBC)[0], axis=1)
    ok = all(dq[j] >= (CLEAR_H if sym_s[j] == "H" else CLEAR_HEAVY) for j in range(ns) if j != m)
    if ok:
        anchors_M.append(dict(kind="M", tag=f"{el}{int(m)}", pos=p, dir=d, leave=[], score=mag + 0.05 * pos[m, 2], el=el))
anchors_M.sort(key=lambda a: -a["score"])
anchors_M = anchors_M[:N_M_ANCHORS]

print(f"Surface hydroxyls found: {len(hyd_O)} (anchors kept: {len(anchors_OH)})")
print(f"Open metal sites kept: {len(anchors_M)}")
assert anchors_OH or anchors_M, "No anchors found; relax the site-finding settings."
pool = anchors_OH + anchors_M

# acceptor oxygens for the P-OH...O(ITO) hydrogen bond (surface oxygens near the top)
acceptors = [int(o) for o in O_idx_s if pos[o, 2] > ztop - 2.0]

def apos(a, ref):
    return unwrap_to(ref, a["pos"])

def adist(a, b):
    return np.linalg.norm(mic_vec(a["pos"], b["pos"]))

def leaves_overlap(*anc):
    s = []
    for a in anc:
        s += a["leave"]
    return len(s) != len(set(s))

def nwater(*anc):
    return sum(a["kind"] == "OH" for a in anc)

# ---- choose multidentate anchor sets, balanced over water-count classes ----
def best_per_class(combos, n_per, nanch):
    out = []
    for nw in range(0, 3):
        sel = sorted([c for c in combos if nwater(*c[1:]) == nw], key=lambda c: c[0])[:n_per]
        out += sel
    return out

pairs = []
for a, b in itertools.combinations(pool, 2):
    d = adist(a, b)
    if OO_MIN <= d <= OO_MAX and not leaves_overlap(a, b):
        pairs.append((abs(d - 2.9), a, b))
pairs = best_per_class(pairs, N_PER_CLASS_BI, 2)

triples = []
for a, b, c in itertools.combinations(pool, 3):
    if nwater(a, b, c) > 2 or all(x["kind"] == "OH" for x in (a, b, c)):
        continue            # at most two P-OH can condense; the P=O oxygen needs an open M site
    ds = [adist(a, b), adist(a, c), adist(b, c)]
    if all(OO_MIN <= d <= OO_MAX for d in ds) and not leaves_overlap(a, b, c):
        triples.append((sum(abs(d - 3.0) for d in ds), a, b, c))
triples = best_per_class(triples, N_PER_CLASS_TRI, 3)

mono_anch = anchors_OH[:N_MONO] + anchors_M[:N_MONO]
hb_anch = anchors_OH[:N_HBOND] + anchors_M[:N_HBOND]
print("Monodentate anchors :", [a["tag"] for a in mono_anch])
print("Bidentate pairs     :", [(a["tag"], b["tag"], nwater(a, b)) for _, a, b in pairs])
print("Tridentate triples  :", [(a["tag"], b["tag"], c["tag"], nwater(a, b, c)) for _, a, b, c in triples])

# ================= generic helpers =================
def kabsch(P, Q, w):
    w = np.asarray(w, float)
    cp = (P * w[:, None]).sum(0) / w.sum()
    cq = (Q * w[:, None]).sum(0) / w.sum()
    H = (P - cp).T @ ((Q - cq) * w[:, None])
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, cp, cq

def P_C_targets(O_targets):
    T = np.array(O_targets)
    cxy = T[:, :2].mean(0)
    r = np.linalg.norm(T[:, :2] - cxy, axis=1).mean()
    dz = np.sqrt(max(D_PO**2 - r**2, 0.5**2))
    Pt = np.array([cxy[0], cxy[1], T[:, 2].mean() + dz])
    Ct = Pt + np.array([0, 0, D_PC])
    return Pt, Ct

def slab_xyz(leave):
    mask = np.ones(ns, bool)
    mask[list(leave)] = False
    return pos[mask]

def min_d(A, B):
    return get_distances(A, B, cell=CELL, pbc=PBC)[1].min()

# ================= per-molecule builder =================
def process(label, mol_file):
    out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)
    mol = read(mol_file)
    mol.set_pbc(False)
    nm = len(mol)

    sym = np.array(mol.get_chemical_symbols())
    iP = int(np.where(sym == "P")[0][0])
    dP = np.linalg.norm(mol.positions - mol.positions[iP], axis=1)
    O_idx = [int(i) for i in np.where(sym == "O")[0] if dP[i] < 2.0]
    iC = int([i for i in np.where(sym == "C")[0] if dP[i] < 2.2][0])
    assert len(O_idx) == 3, f"{label}: expected 3 O on P, found {len(O_idx)} (is the molecule unwrapped?)"

    H_of = {}
    for o in O_idx:
        do = np.linalg.norm(mol.positions - mol.positions[o], axis=1)
        H_of[o] = [int(h) for h in np.where(sym == "H")[0] if do[h] < 1.2]
    oh_O = [o for o in O_idx if H_of[o]]
    po_O = [o for o in O_idx if not H_of[o]]
    assert len(oh_O) == 2 and len(po_O) == 1, f"{label}: expected 2 P-OH and 1 P=O"
    print(f"\n[{label}] P: {iP}  C: {iC}  P-OH O: {oh_O}  P=O O: {po_O}")

    def fit(idx, targets, weights):
        P = mol.positions[idx]
        Q = np.array(targets)
        R, cp, cq = kabsch(P, Q, weights)
        new = (mol.positions - cp) @ R.T + cq
        rmsd = np.sqrt(np.average(((new[idx] - Q) ** 2).sum(1), weights=weights))
        return new, rmsd

    def clash(new_pos, exclude, sxyz):
        rows = [i for i in range(nm) if i not in exclude]
        return min_d(new_pos[rows], sxyz)

    def acid_H_removed(bound, anc_of):
        """P-OH hydrogens lost as water: only for O condensed onto a surface hydroxyl."""
        return {h for o in bound if anc_of[o]["kind"] == "OH" for h in H_of[o]}

    def leaving(anc_of):
        s = set()
        for a in anc_of.values():
            s |= set(a["leave"])
        return s

    def assign(O_list, anchors, weights, constrain_PO=False):
        """Best O -> anchor assignment by Kabsch rmsd. P=O oxygen must go to an open M site."""
        best = None
        ref = anchors[0]["pos"]
        T = [apos(a, ref) for a in anchors]
        Pt, Ct = P_C_targets(T)
        for perm in itertools.permutations(range(len(anchors))):
            if constrain_PO and any(O_list[p] in po_O and anchors[perm[p]]["kind"] != "M" for p in range(len(O_list))):
                continue
            tg = [T[perm[p]] for p in range(len(O_list))]
            new, rmsd = fit(list(O_list) + [iP, iC], tg + [Pt, Ct], list(weights) + [1.0, 1.0])
            if best is None or rmsd < best[1]:
                best = (new, rmsd, {O_list[p]: anchors[perm[p]] for p in range(len(O_list))})
        return best

    def mono_scan(anchor, tilts=(0.0,), psis=(0.0,), extra_score=None):
        """Place O_b on the anchor; scan P-O azimuth (and optional tilt) for the best geometry."""
        O_b = oh_O[0]
        sp = anchor["pos"]
        leave = set(anchor["leave"])
        sxyz = slab_xyz(leave)
        anc_of = {O_b: anchor}
        drop = acid_H_removed([O_b], anc_of)
        excl = {O_b} | drop
        best = None
        for phi in np.deg2rad(np.arange(0, 360, 30)):
            d_PO = np.array([np.sin(np.deg2rad(109.5)) * np.cos(phi),
                             np.sin(np.deg2rad(109.5)) * np.sin(phi),
                             np.cos(np.deg2rad(109.5))])
            Pt = sp - D_PO * d_PO
            for tilt in tilts:
                for psi in (psis if tilt > 0 else (0.0,)):
                    t, ps = np.deg2rad(tilt), np.deg2rad(psi)
                    cdir = np.array([np.sin(t) * np.cos(ps), np.sin(t) * np.sin(ps), np.cos(t)])
                    Ct = Pt + D_PC * cdir
                    new, rmsd = fit([O_b, iP, iC], [sp, Pt, Ct], [1, 1, 1])
                    c = clash(new, excl, sxyz)
                    sc = c if extra_score is None else extra_score(new, c, anc_of, leave)
                    if best is None or sc > best[0]:
                        best = (sc, new, rmsd)
        return best[1], [O_b], anc_of, best[2]

    # ---- mode (a): monodentate ----
    def mono(anchor):
        return mono_scan(anchor)

    # ---- mode (b): one P-O-M bond + P-OH...O(ITO) hydrogen bond ----
    def hbond(anchor):
        O_h = oh_O[1]
        H_i = H_of[O_h][0]
        def score(new, c, anc_of, leave):
            acc = [j for j in acceptors if j not in leave]
            dHA = min_d(new[[H_i]], pos[acc])
            pen = 0.0 if (c >= 1.5 and dHA >= 1.4) else 10.0
            return -abs(dHA - D_HBOND) - pen
        new, bound, anc_of, rmsd = mono_scan(anchor, tilts=(0.0, 25.0, 50.0), psis=np.arange(0, 360, 45), extra_score=score)
        return new, bound, anc_of, rmsd

    # ---- mode (c): bidentate ----
    def bidentate(a, b):
        new, rmsd, anc_of = assign(oh_O, [a, b], [1, 1])
        return new, list(oh_O), anc_of, rmsd

    # ---- mode (d): tridentate ----
    def tridentate(a, b, c):
        Ol = [oh_O[0], oh_O[1], po_O[0]]
        new, rmsd, anc_of = assign(Ol, [a, b, c], [1, 1, 1], constrain_PO=True)
        return new, Ol, anc_of, rmsd

    jobs = [(f"a-mono_{a['tag']}", (lambda a=a: mono(a))) for a in mono_anch]
    jobs += [(f"b-bi-Hbond_{a['tag']}", (lambda a=a: hbond(a))) for a in hb_anch]
    jobs += [(f"c-bi_{a['tag']}-{b['tag']}", (lambda a=a, b=b: bidentate(a, b))) for _, a, b in pairs]
    jobs += [(f"d-tri_{a['tag']}-{b['tag']}-{c['tag']}", (lambda a=a, b=b, c=c: tridentate(a, b, c))) for _, a, b, c in triples]

    summary = []
    for mode, builder in jobs:
        new_pos, bound, anc_of, rmsd = builder()
        leave = leaving(anc_of)
        drop = acid_H_removed(bound, anc_of)
        nw = int(sum(anc_of[o]["kind"] == "OH" for o in bound))
        exclude = set(bound) | drop
        sxyz = slab_xyz(leave)

        lift = 0.0
        while clash(new_pos + np.array([0, 0, lift]), exclude, sxyz) < MIN_CONTACT and lift < MAX_LIFT:
            lift += LIFT_STEP
        placed = new_pos + np.array([0, 0, lift])
        contact = clash(placed, exclude, sxyz)

        # H...O distance for mode (b)
        dHA = np.nan
        if mode.startswith("b-"):
            H_i = H_of[oh_O[1]][0]
            acc = [j for j in acceptors if j not in leave]
            dHA = min_d(placed[[H_i]], pos[acc])

        m = mol.copy()
        m.positions = placed
        if drop:
            m = m[[i for i in range(nm) if i not in drop]]
        s_r = slab[[i for i in range(ns) if i not in leave]]
        s2 = s_r.copy()
        s2 += m
        nsr = len(s_r)
        zmax = s2.positions[:, 2].max()
        c_new = max(slab.cell[2, 2], zmax + VACUUM_TOP)
        cell = s2.cell.array.copy(); cell[2] = [0, 0, c_new]
        s2.set_cell(cell, scale_atoms=False)
        fix_idx = [i for i in range(nsr) if FREEZE_BELOW > 0 and s2.positions[i, 2] < FREEZE_BELOW]
        s2.set_constraint(FixAtoms(indices=fix_idx))

        Dm = s2.get_all_distances(mic=True)
        Dno = s2.get_all_distances(mic=False)
        dmin_slab = Dm[nsr:, :nsr].min()
        sub, subn = Dm[nsr:, nsr:], Dno[nsr:, nsr:]
        img = sub[sub < subn - 0.01]
        dimg = img.min() if img.size else np.inf

        name = f"POSCAR_{label}_{mode}_{nw}H2O"
        write(os.path.join(out_dir, name), s2, format="vasp", direct=True, vasp5=True, sort=True)
        summary.append((name, nw, rmsd, lift, contact, dmin_slab, dimg, dHA))
    return summary

# ================= run all molecules =================
for label, path in MOLECULES.items():
    summ = process(label, path)
    print(f"{'file':62s} {'H2O':>3s} {'rmsd':>5s} {'lift':>5s} {'d(nonbd)':>8s} {'d(all)':>7s} {'d(img)':>7s} {'d(H..O)':>8s}")
    for n, nw, r, lf, c, d1, d2, dh in summ:
        bad = (c < MIN_CONTACT - 0.05 or d1 < 1.6 or d2 < 2.0 or (not np.isnan(dh) and dh > 2.3))
        dhs = "" if np.isnan(dh) else f"{dh:8.2f}"
        print(f"{n:62s} {nw:3d} {r:5.2f} {lf:5.2f} {c:8.2f} {d1:7.2f} {d2:7.2f} {dhs:>8s}{'  <-- check' if bad else ''}")

Slab: H24In84O144Sn12  highest metal z = 8.39  highest heavy atom z = 9.61
Surface hydroxyls found: 24 (anchors kept: 12)
Open metal sites kept: 8
Monodentate anchors : ['OH100', 'OH98', 'OH125', 'OH119', 'In28', 'In22', 'Sn228', 'In16']
Bidentate pairs     : [('In22', 'Sn229', 0), ('OH87', 'In6', 1), ('OH98', 'In27', 1), ('OH100', 'In6', 1), ('OH87', 'OH117', 2), ('OH86', 'OH102', 2), ('OH91', 'OH87', 2)]
Tridentate triples  : [('OH125', 'In22', 'Sn229', 1), ('OH86', 'OH102', 'In7', 2), ('OH87', 'OH86', 'In6', 2)]

[F5BzPA] P: 16  C: 6  P-OH O: [17, 18]  P=O O: [19]
file                                                           H2O  rmsd  lift d(nonbd)  d(all)  d(img)  d(H..O)
POSCAR_F5BzPA_a-mono_OH100_1H2O                                  1  0.07  0.00     2.14    2.14     inf         
POSCAR_F5BzPA_a-mono_OH98_1H2O                                   1  0.07  0.00     1.93    1.65     inf         
POSCAR_F5BzPA_a-mono_OH125_1H2O                                  1  0.07  0.00     2.00

In [3]:
import shutil, os
from google.colab import files

OUT_ROOT = "/content/POSCAR_PA_on_ITO"    # output folder from the script (contains one subfolder per molecule)

shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)       # creates /content/POSCAR_PA_on_ITO.zip
n_files = sum(len(fs) for _, _, fs in os.walk(OUT_ROOT))
print(f"Zipped {n_files} files")
files.download(OUT_ROOT + ".zip")

Zipped 52 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import shutil, os

OUT_ROOT = "/content/POSCAR_PA_on_ITO"

shutil.rmtree(OUT_ROOT, ignore_errors=True)          # the folder with all POSCARs
if os.path.exists(OUT_ROOT + ".zip"):
    os.remove(OUT_ROOT + ".zip")                     # the zip file
print("Removed:", OUT_ROOT, "and", OUT_ROOT + ".zip")

Removed: /content/POSCAR_PA_on_ITO and /content/POSCAR_PA_on_ITO.zip


# PFBT and FBzS + ITO

In [6]:
import itertools, os
import numpy as np
from ase.io import read, write
from ase.geometry import find_mic, get_distances
from ase.constraints import FixAtoms

# =====================================================================
# Thiols (PFBT, FBzS) on hydroxylated ITO(222). Same slab model and anchor
# search as place_PA_on_ITO.py, with the orientation scan of the Au script.
# Anchor types:
#   "M" : exposed, under-coordinated In/Sn. S placed D_M_S from the cation along
#         its open direction. H policies as in the Au script:
#           "thiolate" = S-H hydrogen removed, "thiol" = intact S-H.
#   "OH": surface hydroxyl, thiol condensation  R-SH + HO-M -> R-S-M + H2O.
#         S replaces the hydroxyl O (moved outward until the mean M-S distance is
#         D_M_S); the hydroxyl (O+H) and the S-H hydrogen are removed as water.
#         Switch off with USE_OH_CONDENSATION = False.
# =====================================================================

# ---------------- user settings ----------------
SLAB_FILE = "/content/CONTCAR"                  # ITO slab
MOLECULES = {
    "PFBT": "/content/CONTCAR_PFBT",   # adjust to your actual file names
    "FBzS": "/content/CONTCAR_FBzS",
}
OUT_ROOT   = "/content/POSCAR_thiol_on_ITO"
METALS     = ("In", "Sn")
D_M_S      = {"In": 2.55, "Sn": 2.45}        # target M-S distance (Angstrom)
USE_OH_CONDENSATION = True
TILTS      = (0.0, 30.0)                     # tilt of the S->C bond from the surface normal (degrees)
PHIS       = tuple(np.arange(0, 360, 60))    # azimuth of the tilt (degrees), used when tilt > 0
PSIS       = tuple(np.arange(0, 360, 30))    # spin about the S->C bond (degrees)
OH_CUTOFF   = 1.20    # O-H distance that defines a surface hydroxyl (A)
MO_CUTOFF   = 2.70    # M-O cutoff for coordination / open direction (A)
SURF_DEPTH  = 2.5     # metals within this depth of the highest metal are surface candidates (A)
MIN_OPEN    = 0.35
MIN_UP      = 0.30
CLEAR_HEAVY = 2.50    # open-site S must be this far from other slab heavy atoms (A)
CLEAR_H     = 2.00    # ... and this far from slab H atoms (A)
N_OH_ANCHORS = 8      # surface hydroxyls used as anchors (highest first)
N_M_ANCHORS  = 8      # open metal sites used as anchors (most exposed first)
VACUUM_TOP = 12.0
FREEZE_BELOW = 3.0    # freeze slab atoms with z below this value (A); 0 = none
# -----------------------------------------------

# ================= slab analysis (done once) =================
slab = read(SLAB_FILE)
slab.set_pbc(True)
pos = slab.positions
sym_s = np.array(slab.get_chemical_symbols())
ns = len(slab)
CELL, PBC = slab.cell, slab.pbc
metal_idx = np.where(np.isin(sym_s, METALS))[0]
O_idx_s = np.where(sym_s == "O")[0]
H_idx_s = np.where(sym_s == "H")[0]
zmetal = pos[metal_idx, 2].max()
print(f"Slab: {slab.get_chemical_formula()}  highest metal z = {zmetal:.2f}")
Dss = slab.get_all_distances(mic=True)

def mic_vec(a, b):
    return find_mic((b - a)[None, :], CELL, PBC)[0][0]

def unwrap_to(ref, p):
    return ref + mic_vec(ref, p)

# ---- anchor type "OH": surface hydroxyls (thiol condensation) ----
anchors = []
if USE_OH_CONDENSATION:
    hyd_O = {}
    for h in H_idx_s:
        o = int(O_idx_s[np.argmin(Dss[h, O_idx_s])])
        if Dss[h, o] < OH_CUTOFF:
            hyd_O[o] = int(h)
    cand = []
    for o, h in hyd_O.items():
        ms = [int(m) for m in metal_idx if Dss[o, m] < MO_CUTOFF]
        if not ms:
            continue
        mp = [unwrap_to(pos[o], pos[m]) for m in ms]
        cen = np.mean(mp, axis=0)
        d = pos[o] - cen
        d = d / np.linalg.norm(d) if np.linalg.norm(d) > 1e-6 else np.array([0, 0, 1.0])
        target = np.mean([D_M_S[sym_s[m]] for m in ms])
        best_t, best_e = 0.0, 1e9
        for t in np.arange(0.0, 2.01, 0.05):          # move S outward until mean M-S = target
            e = abs(np.mean([np.linalg.norm(pos[o] + t * d - q) for q in mp]) - target)
            if e < best_e:
                best_t, best_e = t, e
        p = pos[o] + best_t * d
        dq = np.linalg.norm(find_mic(pos - p, CELL, PBC)[0], axis=1)
        if all(dq[j] >= (1.8 if sym_s[j] == "H" else 2.0) for j in range(ns) if j not in (o, h)):
            cand.append(dict(kind="OH", tag=f"OH{o}", pos=p, dir=d, leave=[o, h], score=pos[o, 2]))
    cand.sort(key=lambda a: -a["score"])
    anchors += cand[:N_OH_ANCHORS]
    print(f"Surface hydroxyls found: {len(hyd_O)} (anchors kept: {min(len(cand), N_OH_ANCHORS)})")

# ---- anchor type "M": exposed, unobstructed under-coordinated cations ----
candM = []
for m in metal_idx:
    if pos[m, 2] < zmetal - SURF_DEPTH:
        continue
    vecs = slab.get_distances(int(m), range(ns), mic=True, vector=True)
    dist = np.linalg.norm(vecs, axis=1)
    nb = [j for j in range(ns) if j != m and sym_s[j] == "O" and dist[j] < MO_CUTOFF]
    if not nb:
        continue
    u = vecs[nb] / dist[nb][:, None]
    ov = -u.sum(axis=0)
    mag = np.linalg.norm(ov)
    if mag < MIN_OPEN:
        continue
    d = ov / mag
    if d[2] < MIN_UP:
        continue
    el = sym_s[m]
    p = pos[m] + D_M_S[el] * d
    dq = np.linalg.norm(find_mic(pos - p, CELL, PBC)[0], axis=1)
    if all(dq[j] >= (CLEAR_H if sym_s[j] == "H" else CLEAR_HEAVY) for j in range(ns) if j != m):
        candM.append(dict(kind="M", tag=f"{el}{int(m)}", pos=p, dir=d, leave=[], score=mag + 0.05 * pos[m, 2]))
candM.sort(key=lambda a: -a["score"])
anchors += candM[:N_M_ANCHORS]
print(f"Open metal sites kept: {min(len(candM), N_M_ANCHORS)}")
assert anchors, "No anchors found; relax the site-finding settings."
print("Anchors:", [a["tag"] for a in anchors])

fix_all = lambda arr: [i for i in range(len(arr)) if FREEZE_BELOW > 0 and arr[i, 2] < FREEZE_BELOW]

# ================= rotation helpers =================
def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def rot_axis(axis, ang):
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    K = skew(axis)
    return np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * (K @ K)

def rot_align(a, b):
    a = a / np.linalg.norm(a); b = b / np.linalg.norm(b)
    v = np.cross(a, b); c = float(np.dot(a, b)); s = np.linalg.norm(v)
    if s < 1e-8:
        if c > 0:
            return np.eye(3)
        ax = np.cross(a, [1, 0, 0])
        if np.linalg.norm(ax) < 1e-6:
            ax = np.cross(a, [0, 1, 0])
        return rot_axis(ax, np.pi)
    K = skew(v)
    return np.eye(3) + K + (K @ K) * ((1 - c) / s**2)

# ================= per-molecule builder =================
def process(label, mol_file):
    out_dir = os.path.join(OUT_ROOT, label)
    os.makedirs(out_dir, exist_ok=True)
    mol = read(mol_file)
    mol.set_pbc(False)

    sym = np.array(mol.get_chemical_symbols())
    assert (sym == "S").sum() == 1, f"{label}: expected exactly one S"
    iS = int(np.where(sym == "S")[0][0])
    dS = np.linalg.norm(mol.positions - mol.positions[iS], axis=1)
    Cs = [int(i) for i in np.where(sym == "C")[0] if dS[i] < 2.0]
    assert len(Cs) == 1, f"{label}: expected one C bonded to S, found {len(Cs)}"
    iC = Cs[0]
    H_S = [int(h) for h in np.where(sym == "H")[0] if dS[h] < 1.6]
    print(f"\n[{label}] S: {iS}  C: {iC}  H on S: {H_S}")

    u0 = mol.positions[iC] - mol.positions[iS]
    summary = []

    for anc in anchors:
        spos = anc["pos"]
        leave = set(anc["leave"])
        keep_idx = [i for i in range(ns) if i not in leave]
        s_r = slab[keep_idx]
        sxyz = s_r.positions
        if anc["kind"] == "OH":
            policies = ("H2O",) if H_S else ()      # condensation needs the S-H hydrogen
        else:
            policies = ("thiolate", "thiol") if H_S else ("thiolate",)
        for tilt in TILTS:
            for policy in policies:
                drop = set(H_S) if policy in ("H2O", "thiolate") else set()
                excl = {iS} | drop
                rows = [i for i in range(len(mol)) if i not in excl]
                best = None
                for phi in (PHIS if tilt > 0 else (0.0,)):
                    t = np.array([np.sin(np.deg2rad(tilt)) * np.cos(np.deg2rad(phi)),
                                  np.sin(np.deg2rad(tilt)) * np.sin(np.deg2rad(phi)),
                                  np.cos(np.deg2rad(tilt))])
                    R1 = rot_align(u0, t)
                    for psi in PSIS:
                        R = rot_axis(t, np.deg2rad(psi)) @ R1
                        newpos = (mol.positions - mol.positions[iS]) @ R.T + spos
                        c = get_distances(newpos[rows], sxyz, cell=CELL, pbc=PBC)[1].min()
                        if best is None or c > best[0]:
                            best = (c, newpos)
                m = mol.copy()
                m.positions = best[1]
                if drop:
                    m = m[[i for i in range(len(m)) if i not in drop]]
                s2 = s_r.copy()
                s2 += m
                nsr = len(s_r)
                zmax = s2.positions[:, 2].max()
                c_new = max(slab.cell[2, 2], zmax + VACUUM_TOP)
                cell = s2.cell.array.copy(); cell[2] = [0, 0, c_new]
                s2.set_cell(cell, scale_atoms=False)
                s2.set_constraint(FixAtoms(indices=fix_all(s2.positions[:nsr])))
                Dm = s2.get_all_distances(mic=True)
                Dno = s2.get_all_distances(mic=False)
                sub, subn = Dm[nsr:, nsr:], Dno[nsr:, nsr:]
                img = sub[sub < subn - 0.01]
                dimg = img.min() if img.size else np.inf
                dMS = get_distances([spos], sxyz, cell=CELL, pbc=PBC)[1].min()
                nw = 1 if policy == "H2O" else 0
                name = f"POSCAR_{label}_{anc['tag']}_tilt{int(tilt)}_{policy}"
                write(os.path.join(out_dir, name), s2, format="vasp", direct=True, vasp5=True, sort=True)
                summary.append((name, nw, dMS, best[0], dimg))
    return summary

for label, path in MOLECULES.items():
    summ = process(label, path)
    print(f"{'file':50s} {'H2O':>3s} {'d(M-S)':>7s} {'min d(other-slab)':>18s} {'min d(image)':>13s}")
    for n, nw, d0, d1, d2 in summ:
        flag = "  <-- check" if (d1 < 2.0 or d2 < 2.0 or d0 < 2.0) else ""
        print(f"{n:50s} {nw:3d} {d0:7.2f} {d1:18.2f} {d2:13.2f}{flag}")

Slab: H24In84O144Sn12  highest metal z = 8.39
Surface hydroxyls found: 24 (anchors kept: 8)
Open metal sites kept: 8
Anchors: ['OH100', 'OH119', 'OH101', 'OH91', 'OH87', 'OH99', 'OH86', 'OH102', 'In28', 'In22', 'In16', 'In6', 'In27', 'Sn229', 'In7', 'In26']

[PFBT] S: 11  C: 2  H on S: [12]
file                                               H2O  d(M-S)  min d(other-slab)  min d(image)
POSCAR_PFBT_OH100_tilt0_H2O                          1    2.55               3.56           inf
POSCAR_PFBT_OH100_tilt30_H2O                         1    2.55               3.76           inf
POSCAR_PFBT_OH119_tilt0_H2O                          1    2.42               2.61           inf
POSCAR_PFBT_OH119_tilt30_H2O                         1    2.42               3.24           inf
POSCAR_PFBT_OH101_tilt0_H2O                          1    1.94               2.44           inf  <-- check
POSCAR_PFBT_OH101_tilt30_H2O                         1    1.94               2.72           inf  <-- check
POSCAR_PFBT_OH

In [7]:
import shutil, os
from google.colab import files

OUT_ROOT = "/content/POSCAR_thiol_on_ITO"    # output folder from the script (contains one subfolder per molecule)

shutil.make_archive(OUT_ROOT, "zip", OUT_ROOT)       # creates /content/POSCAR_thiol_on_ITO.zip
n_files = sum(len(fs) for _, _, fs in os.walk(OUT_ROOT))
print(f"Zipped {n_files} files")
files.download(OUT_ROOT + ".zip")

Zipped 96 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import shutil, os

OUT_ROOT = "/content/POSCAR_thiol_on_ITO"

shutil.rmtree(OUT_ROOT, ignore_errors=True)          # the folder with all POSCARs
if os.path.exists(OUT_ROOT + ".zip"):
    os.remove(OUT_ROOT + ".zip")                     # the zip file
print("Removed:", OUT_ROOT, "and", OUT_ROOT + ".zip")

Removed: /content/POSCAR_thiol_on_ITO and /content/POSCAR_thiol_on_ITO.zip
